In [1]:
# SDK模型下载
from modelscope import snapshot_download
cosyvoice_root_dir = '/home/zhangjiayuan/program/audio/tts/CosyVoice'
model_dir='/data/nas/models/tts/CosyVoice'

# snapshot_download('iic/CosyVoice2-0.5B', local_dir=f'{model_dir}/pretrained_models/CosyVoice2-0.5B')
# snapshot_download('iic/CosyVoice-300M', local_dir=f'{model_dir}/pretrained_models/CosyVoice-300M')
# snapshot_download('iic/CosyVoice-300M-25Hz', local_dir=f'{model_dir}/pretrained_models/CosyVoice-300M-25Hz')
# snapshot_download('iic/CosyVoice-300M-SFT', local_dir=f'{model_dir}/pretrained_models/CosyVoice-300M-SFT')
# snapshot_download('iic/CosyVoice-300M-Instruct', local_dir=f'{model_dir}/pretrained_models/CosyVoice-300M-Instruct')
# snapshot_download('iic/CosyVoice-ttsfrd', local_dir=f'{model_dir}/pretrained_models/CosyVoice-ttsfrd')

2025-02-26 02:19:15,173 - modelscope - INFO - PyTorch version 2.3.1+cu121 Found.
2025-02-26 02:19:15,176 - modelscope - INFO - Loading ast index from /home/zhangjiayuan/.cache/modelscope/ast_indexer
2025-02-26 02:19:15,298 - modelscope - INFO - Loading done! Current index file version is 1.15.0, with md5 d62d820367ec37a8939cca3270a4c57f and a total number of 980 components indexed
/home/zhangjiayuan/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.path.append(f'{cosyvoice_root_dir}')
sys.path.append(f'{cosyvoice_root_dir}/third_party/Matcha-TTS')
from cosyvoice.cli.cosyvoice import CosyVoice, CosyVoice2
from cosyvoice.utils.file_utils import load_wav
import torchaudio

# CosyVoice2 Usage

In [4]:
cosyvoice = CosyVoice2(f'{model_dir}/pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=False)

# NOTE if you want to reproduce the results on https://funaudiollm.github.io/cosyvoice2, please add text_frontend=False during inference
# zero_shot usage
prompt_speech_16k = load_wav('../asset/zero_shot_prompt.wav', 16000)
for i, j in enumerate(cosyvoice.inference_zero_shot('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', '希望你以后能够做的比我还好呦。', prompt_speech_16k, stream=False)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

# fine grained control, for supported control, check cosyvoice/tokenizer/tokenizer.py#L248
for i, j in enumerate(cosyvoice.inference_cross_lingual('在他讲述那个荒诞故事的过程中，他突然[laughter]停下来，因为他自己也被逗笑了[laughter]。', prompt_speech_16k, stream=False)):
    torchaudio.save('fine_grained_control_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

# instruct usage
for i, j in enumerate(cosyvoice.inference_instruct2('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', '用四川话说这句话', prompt_speech_16k, stream=False)):
    torchaudio.save('instruct_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

2025-02-18 08:34:19,598 INFO input frame rate=25
2025-02-18 08:34:21.921622018 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 8 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-02-18 08:34:21.927826609 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-02-18 08:34:21.927842445 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
festival_initialize() called more than once


open voice lang map failed


  0%|          | 0/1 [00:00<?, ?it/s]2025-02-18 08:34:38,305 INFO synthesis text 收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-02-18 08:35:01,351 INFO yield speech len 12.32, rtf 1.8706396996200858
  0%|          | 0/1 [00:00<?, ?it/s]2025-02-18 08:35:01,825 INFO synthesis text 在他讲述那个荒诞故事的过程中，他突然[laughter]停下来，因为他自己也被逗笑了[laughter]。
2025-02-18 08:35:18,086 INFO yield speech len 8.96, rtf 1.81491132825613
  0%|          | 0/1 [00:00<?, ?it/s]2025-02-18 08:35:18,306 INFO synthesis text 收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-02-18 08:35:36,775 INFO yield speech len 10.44, rtf 1.7690151587299918
100%|██████████| 1/1 [00:18<00:00, 18.60s/it]


# 推理使用 spk

In [4]:
cosyvoice = CosyVoice2(f'{model_dir}/pretrained_models/CosyVoice2-0.5B', load_jit=False, load_trt=False, fp16=False)

# NOTE if you want to reproduce the results on https://funaudiollm.github.io/cosyvoice2, please add text_frontend=False during inference
# zero_shot usage
prompt_speech_16k = load_wav('../asset/zero_shot_prompt.wav', 16000)
# for i, j in enumerate(cosyvoice.inference_zero_shot('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', '希望你以后能够做的比我还好呦。', prompt_speech_16k, stream=False)):
#     torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    
spk_data = cosyvoice.generate_spk_data('希望你以后能够做的比我还好呦。', prompt_speech_16k)
text = '国家统计局今天早上是发布了三月份以及一季度的相应的经济数据。整体上数据呢还是延续一个稳步复苏的一个态势。宏观经济呢是没有太大的一个像下行的一个风险。国家统计局今天早上是发布了三月份以及一季度的相应的经济数据。整体上数据呢还是延续一个稳步复苏的一个态势。宏观经济呢是没有太大的一个像下行的一个风险。'
speech_datas = []
for i, j in enumerate(cosyvoice.inference_zero_shot_with_spk(text, spk_data, stream=False)):
    print(j.keys())
    speech_datas.append(j['tts_speech'])
    torchaudio.save('zero_shot_spk_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)
    print(speech_datas[-1].shape, type(speech_datas[-1]))
    

/home/zhangjiayuan/miniconda3/envs/cosyvoice/lib/python3.10/site-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)
2025-02-26 02:24:03,827 INFO input frame rate=25


open voice lang map failed


2025-02-26 02:24:06.208260557 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 8 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-02-26 02:24:06.215042217 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-02-26 02:24:06.215064706 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
festival_initialize() called more than once
  0%|          | 0/2 [00:00<?, ?it/s]2025-02-26 02:30:30,963 INFO synthesis text 国家统计局今天早上是发布了三月份以及一季度的相应的经济数据。整体上数据呢还是延续一个稳步复苏的一个态势。宏观经济呢是没有太大的一个像下行

dict_keys(['tts_speech', 'norm_text'])
torch.Size([1, 428160]) <class 'torch.Tensor'>


2025-02-26 02:31:06,976 INFO yield speech len 19.28, rtf 0.9318071158595105
100%|██████████| 2/2 [03:39<00:00, 109.73s/it]

dict_keys(['tts_speech', 'norm_text'])
torch.Size([1, 462720]) <class 'torch.Tensor'>


In [10]:
import torch
concatenated_tensor_dim0 = torch.cat(speech_datas, dim=1)
print(concatenated_tensor_dim0.shape)

torch.Size([1, 887040])


# CosyVoice Usage

In [4]:
cosyvoice = CosyVoice(f'{model_dir}/pretrained_models/CosyVoice-300M-SFT', load_jit=False, load_trt=False, fp16=False)
# sft usage
print(cosyvoice.list_available_spks())
# change stream=True for chunk stream inference
for i, j in enumerate(cosyvoice.inference_sft('你好，我是通义生成式语音大模型，请问有什么可以帮您的吗？', '中文女', stream=False)):
    torchaudio.save('sft_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)


cosyvoice = CosyVoice(f'{model_dir}/pretrained_models/CosyVoice-300M') # or change to pretrained_models/CosyVoice-300M-25Hz for 25Hz inference
# zero_shot usage, <|zh|><|en|><|jp|><|yue|><|ko|> for Chinese/English/Japanese/Cantonese/Korean
prompt_speech_16k = load_wav('../asset/zero_shot_prompt.wav', 16000)
for i, j in enumerate(cosyvoice.inference_zero_shot('收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。', '希望你以后能够做的比我还好呦。', prompt_speech_16k, stream=False)):
    torchaudio.save('zero_shot_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)


# cross_lingual usage
prompt_speech_16k = load_wav('../asset/cross_lingual_prompt.wav', 16000)
for i, j in enumerate(cosyvoice.inference_cross_lingual('<|en|>And then later on, fully acquiring that company. So keeping management in line, interest in line with the asset that\'s coming into the family is a reason why sometimes we don\'t buy the whole thing.', prompt_speech_16k, stream=False)):
    torchaudio.save('cross_lingual_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)


# vc usage
prompt_speech_16k = load_wav('../asset/zero_shot_prompt.wav', 16000)
source_speech_16k = load_wav('../asset/cross_lingual_prompt.wav', 16000)
for i, j in enumerate(cosyvoice.inference_vc(source_speech_16k, prompt_speech_16k, stream=False)):
    torchaudio.save('vc_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

cosyvoice = CosyVoice(f'{model_dir}/pretrained_models/CosyVoice-300M-Instruct')
# instruct usage, support <laughter></laughter><strong></strong>[laughter][breath]
for i, j in enumerate(cosyvoice.inference_instruct('在面对挑战时，他展现了非凡的<strong>勇气</strong>与<strong>智慧</strong>。', '中文男', 'Theo \'Crimson\', is a fiery, passionate rebel leader. Fights with fervor for justice, but struggles with impulsiveness.', stream=False)):
    torchaudio.save('instruct_{}.wav'.format(i), j['tts_speech'], cosyvoice.sample_rate)

2025-02-18 05:51:37,091 INFO input frame rate=50
2025-02-18 05:51:39.563325853 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 12 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-02-18 05:51:39.566201882 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-02-18 05:51:39.566217364 [W:onnxruntime:, session_state.cc:1168 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
festival_initialize() called more than once


open voice lang map failed
['中文女', '中文男', '日语男', '粤语女', '英文女', '英文男', '韩语女']


  0%|          | 0/1 [00:00<?, ?it/s]2025-02-18 05:51:45,141 INFO synthesis text 你好，我是通义生成式语音大模型，请问有什么可以帮您的吗？
2025-02-18 05:51:50,833 INFO yield speech len 5.0967800453514736, rtf 1.116710115576286
100%|██████████| 1/1 [00:05<00:00,  5.91s/it]
2025-02-18 05:51:59,340 INFO input frame rate=50
2025-02-18 05:52:01.747133060 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 12 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-02-18 05:52:01.749996530 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-02-18 05:52:01.750012074 [W:onnxruntime:, session_state.cc:1168 VerifyEachNo

open voice lang map failed


  0%|          | 0/1 [00:00<?, ?it/s]2025-02-18 05:52:07,774 INFO synthesis text 收到好友从远方寄来的生日礼物，那份意外的惊喜与深深的祝福让我心中充满了甜蜜的快乐，笑容如花儿般绽放。
2025-02-18 05:52:19,162 INFO yield speech len 11.77251700680272, rtf 0.9672960624671899
  0%|          | 0/1 [00:00<?, ?it/s]

min value is  tensor(-1.0054)


2025-02-18 05:52:20,025 INFO synthesis text <|en|>And then later on, fully acquiring that company. So keeping management in line, interest in line with the asset that's coming into the family is a reason why sometimes we don't buy the whole thing.
2025-02-18 05:52:31,715 INFO yield speech len 11.38938775510204, rtf 1.0263502567903984
100%|██████████| 1/1 [00:12<00:00, 12.03s/it]
2025-02-18 05:52:32,877 INFO yield speech len 13.734603174603174, rtf 0.07448703237980075
2025-02-18 05:52:41,419 INFO input frame rate=50
2025-02-18 05:52:44.495671257 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 12 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2025-02-18 05:52:44.498506415 [W:onnxruntime:, session_state.cc:1166 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred exec

open voice lang map failed


  0%|          | 0/1 [00:00<?, ?it/s]2025-02-18 05:52:56,407 INFO synthesis text 在面对挑战时，他展现了非凡的<strong>勇气</strong>与<strong>智慧</strong>。
2025-02-18 05:53:02,365 INFO yield speech len 5.491519274376417, rtf 1.0848925517868038
100%|██████████| 1/1 [00:06<00:00,  6.00s/it]


In [5]:
print(prompt_speech_16k.squeeze().shape)

torch.Size([55726])
